# BTL Học Máy: Tái hiện Lightweight YOLOv4 Helmet Detection (MDPI Sensors 2023)
Notebook huấn luyện phân tán 6 biến thể mô hình (Model-1 đến Model-6) trên Google Colab.
Tuân thủ quy trình 2 giai đoạn: 50 epochs Freeze Backbone (lr=0.001) + 150 epochs Unfreeze toàn mạng (lr=0.0001).

## 1. Kiểm tra phần cứng GPU

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

## 2. Gắn kết Google Drive (Lưu Checkpoint phòng ngừa Timeout)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone Repository & Cài đặt Thư viện

In [ ]:
!git clone https://github.com/ptanh05/BTL-ML.git
%cd BTL-ML
!pip install -r requirements.txt

## 4. Tăng tốc I/O: Giải nén dữ liệu vào SSD cục bộ Colab

In [ ]:
import os, zipfile
zip_path = '/content/drive/MyDrive/BTL_ML/Dataset_Split-20261005T022535Z-1-001.zip'
extract_path = '/content/dataset'
if os.path.exists(zip_path) and not os.path.exists(extract_path):
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print('>> Đã giải nén dữ liệu vào SSD cục bộ!')
else:
    print('>> Dữ liệu đã sẵn sàng!')

## 5. Khởi chạy Huấn luyện (Phân công Model-1 đến Model-6)
- Sơn: configs/model1.yaml
- Hưng: configs/model2.yaml hoặc configs/model3.yaml
- Tiến: configs/model4.yaml
- T.A: configs/model5.yaml
- Tú: configs/model6.yaml

In [ ]:
# Chọn file cấu hình model mà bạn phụ trách:
CONFIG_FILE = 'configs/model6.yaml'
!python train.py --config $CONFIG_FILE --data configs/dataset.yaml

## 6. Tiếp tục Huấn luyện khi bị ngắt phiên (Resume Checkpoint)

In [ ]:
CHECKPOINT_PATH = '/content/drive/MyDrive/BTL_ML/checkpoints/model6_last.pth'
!python train.py --config $CONFIG_FILE --data configs/dataset.yaml --resume $CHECKPOINT_PATH

## 7. Đánh giá Chỉ số Nghiệm thu (Benchmark mAP & FPS)

In [ ]:
WEIGHTS_PATH = 'weights/model6_best.pth'
!python eval.py --weights $WEIGHTS_PATH --config $CONFIG_FILE --data configs/dataset.yaml

## 8. Suy luận Nhận diện Mũ bảo hộ trên ảnh thực tế (Demo)

In [ ]:
!python detect.py --weights $WEIGHTS_PATH --config $CONFIG_FILE --source dataset/test/ --output results/detections/